# ECE457B Assignment 3 - Question 3

Name: Michael McPhee  
Waterloo ID: 21021347

All random operations in this notebook use seed `123123`, as required.

## Q3.1 - Token embeddings

In [1]:
import torch
import torch.nn as nn

torch.manual_seed(123123)
torch.set_printoptions(linewidth=80)

def get_token_embeddings(sentence):
    """Return a 16-dimensional embedding for each space-separated word."""
    if sentence.endswith('.'):
        sentence = sentence[:-1]
    words = sentence.split(' ')
    distinct_words = sorted(set(words))
    word_to_index = {word: index for index, word in enumerate(distinct_words, start=1)}
    token_indices = torch.tensor([word_to_index[word] for word in words], dtype=torch.long)
    embedding = nn.Embedding(len(distinct_words) + 1, 16)
    return embedding(token_indices)

sentence = 'Attention is all you need for now.'
X = get_token_embeddings(sentence)
print(X)
print('Q3.1 embedding shape:', tuple(X.shape))

tensor([[ 0.5449,  1.9281, -0.6208,  1.1076,  0.1331,  0.1472, -1.0472,  1.1448,
          0.0128, -0.2665,  0.9189, -2.2912,  0.6965,  1.0309, -1.5584, -1.3782],
        [ 0.7076,  1.1984, -1.5267, -0.2254,  0.0264,  0.6631, -0.3922, -0.2221,
          0.5445, -0.3446, -0.8801, -0.5930, -0.0471,  2.0033, -0.9231, -0.3465],
        [ 0.2683,  0.8987,  0.1029,  0.8214,  2.5925,  1.0087, -0.5200,  0.1927,
          0.6977,  0.4074,  0.9424, -1.7282,  1.1603, -1.2132, -1.4624,  0.2962],
        [ 0.3450,  0.2975,  0.9669, -1.4826,  1.0491, -0.6034, -0.2254,  0.1509,
          0.4969,  0.3232, -0.4563, -0.1595,  0.7242, -0.9060, -0.4738,  0.6987],
        [ 0.0381,  0.1548, -0.2557,  0.0346,  0.3751, -0.3636, -2.5965,  0.5740,
          0.3037,  0.4757, -0.0147, -0.4754, -1.6769,  1.5561,  0.3053, -3.1729],
        [ 0.1769, -0.9805, -2.3564, -0.0269, -1.3488,  1.0367,  0.7383,  0.9799,
         -0.0074, -0.1416, -0.7462,  0.3935,  0.6559, -0.5849,  0.9106, -1.3100],
        [-0.7573, -0.1

The final period is removed before splitting, so the sentence contains seven tokens.
The dictionary indices begin at 1, leaving row 0 of the embedding table unused.

## Q3.2 - Single-head attention

In [2]:
def attention(X, q, k, v):
    """Compute scaled dot-product attention for an input X."""
    query = X @ q.T
    key = X @ k.T
    value = X @ v.T
    alignment_scores = query @ key.T
    attention_weights = torch.softmax(
        alignment_scores / (k.shape[0] ** 0.5), dim=-1
    )
    return attention_weights @ value

torch.manual_seed(123123)
q = torch.randn(24, 16)
k = torch.randn(24, 16)
v = torch.randn(28, 16)
C = attention(X, q, k, v)
print(C)
print('Q3.2 context shape:', tuple(C.shape))

tensor([[-3.9901,  4.3275, -4.7719,  4.1242,  1.6817, -0.7436,  7.9075, -4.4122,
         -2.9055, -1.9920,  0.9243,  5.8541,  5.0291, -0.6641,  0.6263, -7.0567,
         -4.3895,  1.8412, -6.6328, -5.0559,  1.8603, -0.6992, -3.3047, -6.4059,
         -0.8336, -1.7719,  5.3621,  2.7600],
        [-3.9901,  4.3275, -4.7719,  4.1242,  1.6817, -0.7436,  7.9075, -4.4122,
         -2.9055, -1.9920,  0.9243,  5.8541,  5.0291, -0.6641,  0.6263, -7.0567,
         -4.3895,  1.8412, -6.6328, -5.0559,  1.8603, -0.6992, -3.3048, -6.4059,
         -0.8336, -1.7719,  5.3621,  2.7600],
        [-3.9901,  4.3275, -4.7719,  4.1242,  1.6817, -0.7436,  7.9075, -4.4122,
         -2.9055, -1.9920,  0.9243,  5.8541,  5.0291, -0.6641,  0.6263, -7.0567,
         -4.3895,  1.8412, -6.6328, -5.0559,  1.8603, -0.6992, -3.3047, -6.4059,
         -0.8336, -1.7719,  5.3621,  2.7600],
        [ 4.8897,  3.0723, -3.8061, -1.9477,  4.1149, -3.6822, -0.3399, -0.4775,
         -0.6445,  1.8027, -0.4289, -2.2086, -8.1640

The query and key projections both have dimension 24, so their dot products
produce a 7 by 7 alignment matrix. The value projection has dimension 28,
giving one 28-dimensional context vector per token.

## Q3.3 - Multi-headed attention

In [3]:
def multi_head_attention(X, h, dq=24, dk=24, dv=28):
    """Concatenate h attention heads and project back to embedding size."""
    head_contexts = []
    for _ in range(h):
        q_head = torch.randn(dq, X.shape[1])
        k_head = torch.randn(dk, X.shape[1])
        v_head = torch.randn(dv, X.shape[1])
        head_contexts.append(attention(X, q_head, k_head, v_head))
    concatenated = torch.cat(head_contexts, dim=-1)
    projection = torch.randn(X.shape[1], h * dv)
    return concatenated @ projection.T

torch.manual_seed(123123)
multi_head_context = multi_head_attention(X, h=5)
print(multi_head_context)
print('Q3.3 context shape:', tuple(multi_head_context.shape))

tensor([[ -48.7671,   10.4616,  -25.6981,   -0.1334,   15.0412,    9.7508,
            4.2829,    3.2145,  -10.9791,  -10.6503,   57.4837,   -3.5944,
           66.4321,   27.8822,   79.1775,    0.8118],
        [   6.4465,   45.1677,  -10.1596,   42.4133,  -17.2532,   35.5593,
          -29.4919,    8.9377,  -29.2499,   39.5382,   14.9105,   39.3742,
           89.2311,   15.8387,  -33.2099,  -31.5830],
        [ -47.6705,   65.0434,   51.0552,   29.2642,  -18.4411,   -1.5422,
            0.8786,    7.3891,  -14.8296,  -11.1926,   41.3006,   11.0176,
           57.2607,   -5.4784,    0.3422,   25.1532],
        [ -29.0072,   63.0510,   50.3895,  -25.4592,   49.0325,  -18.7994,
          -20.5818,   56.3604,  -12.5766,  -15.3697,   11.0065,  -14.7015,
           -0.7090,  -15.6674,  -18.8309,   18.9879],
        [  58.2090,   -6.7888,   -0.3748,   10.8923,    4.9321,  -12.6637,
          -40.1645,  -32.6318,  -29.6094,  -46.0205,  -27.5995,  -10.1244,
           16.9396, -125.7164,  -8

Each of the five heads produces a 7 by 28 context matrix. Concatenating them
gives 7 by 140, and the learned linear projection maps this back to the
original embedding dimension, producing shape `(7, 16)`.

## Q3.4 - Cross-attention

Cross-attention lets the queries come from one sequence while the keys and
values come from another. Compared with self-attention, this allows each
query token to retrieve and combine information from a separate source
sequence rather than only mixing information within its own sequence.

It is useful when the sequences have different roles, such as a decoder
attending to an encoded source sentence in machine translation,
an image-captioning decoder attending to image features, or a
text-conditioned diffusion model attending to text embeddings. Multiple
heads can learn different relationships between the two sequences in parallel.

## Q3.5

Q3.5 is completed in the separate
[A3-Q3.5.ipynb](./A3-Q3.5.ipynb), as required by the assignment.